<a href="https://colab.research.google.com/github/nepslor/teaching/blob/main/CAS_BDML/TS_forecasting_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb


df = pd.read_pickle('https://github.com/nepslor/teaching/raw/refs/heads/main/TimeSeriesForecasting/data/minichallenge/weather.pk')

In [ ]:
df.tail(4)

In [ ]:
ax = df.tail(1000).plot(figsize=(12, 10), subplots=True, linewidth=0.5)
[a.legend(loc='upper right') for a in ax]
plt.show()

In [ ]:
# correlation heatmap sb
fig, ax = plt.subplots(figsize=(10, 10))
sb.heatmap(df.corr(), annot=True, cmap='coolwarm', ax=ax)
plt.show()

In [ ]:
target = 'T (degC)'
df.corr()[target].abs().sort_values(ascending=False)

In [ ]:
selected_vars = df.corr()[target].abs().sort_values(ascending=False).iloc[:-4].index
df = df[selected_vars]

In [ ]:
# pairplot seaborn
sb.pairplot(df.sample(4000), plot_kws={'alpha': 0.5, 's':1}, x_vars=selected_vars[1:], y_vars=[selected_vars[0]])
plt.show()

In [ ]:
for v in selected_vars:
  daily_v = df.assign(                            # assign method temporaly adds new features to a dataframe
      day=df.index.date,
      hour=df.index.hour
  ).pivot(index='hour', columns='day', values=v)  # pivot create a matrix from "index" and "columns"

  # plot heatmap
  fig, ax = plt.subplots(figsize=(15, 2))
  sb.heatmap(daily_v, cmap='viridis', ax=ax)
  ax.set_title('Daily {}'.format(v))


In [ ]:
df = df.ffill().bfill()

In [ ]:
df[target].plot(linewidth=1, figsize=(15, 4))

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(2, 1, figsize=(15, 4), layout='tight')
plot_acf(df[target], lags=24*8, ax=ax[0], linewidth=1);
df[target].tail(24*60).plot(linewidth=1, ax=ax[1])



In [ ]:

from statsmodels.tsa.stattools import adfuller

result = adfuller(df[target].tail(24*30), autolag='AIC')
print('ADF Statistic: %f' % result[0])
print('p-value: %f' % result[1])


result = adfuller(df[target].tail(24*30).diff().dropna(), autolag='AIC')
print('ADF Statistic: %f' % result[0])
print('p-value: %f' % result[1])


The time series is not stationary but it is when defferentiated

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor

import numpy as np



In [ ]:
df_red = df.tail(10000)

tr_ratio = 0.7
n_tr = int(len(df_red) * tr_ratio)

def add_lags(df, vars, lags):
    df_lags = df.copy()
    for v in vars:
        for l in lags:
            df_lags['{}_lag_{}'.format(v, l)] = df[v].shift(l)
    return df_lags.dropna()


# add target transform
y = add_lags(df_red[[target]], [target], -np.arange(1, 25)).iloc[:, 1:]
X = add_lags(df_red, selected_vars[1:], np.hstack([np.arange(1, 25), 24*2]))
df_postprocess = pd.concat({'X':X, 'y':y}, axis=1).dropna()

X_tr, y_tr = df_postprocess['X'].iloc[:n_tr], df_postprocess['y'].iloc[:n_tr]
X_te, y_te = df_postprocess['X'].iloc[n_tr:], df_postprocess['y'].iloc[n_tr:]


lgb_models = [LGBMRegressor(force_col_wise=True,verbose=0).fit(X_tr, y_tr[c].values) for c in y.columns]
y_hat_lgb = pd.DataFrame({f'{target}_lag_{-l}': m.predict(X_te) for l, m in enumerate(lgb_models, 1)}, index=y_te.index)
y_hat_knn = pd.DataFrame(KNeighborsRegressor().fit(X_tr, y_tr.values).predict(X_te), index=y_te.index)


In [ ]:
from matplotlib import animation

def animate_forecasts(y_te, y_hat_lgb, y_hat_knn):
  fig, ax = plt.subplots(figsize=(10, 3))
  offset = 24*7
  def animate(i):
    ax.clear()
    ax.plot(y_te.iloc[i:i+offset + 24, 0].values, label='observed')
    ax.plot(np.arange(offset, offset+24), y_hat_lgb.iloc[i+offset, :].values, label='lgb')
    ax.plot(np.arange(offset, offset+24), y_hat_knn.iloc[i+offset, :].values, label='lin')
    ax.legend(loc='upper left')
    return ax,

  ani = animation.FuncAnimation(fig, animate, frames=np.arange(24*30)[::2], interval=80)
  from IPython.display import HTML
  plt.close(fig)
  return HTML(ani.to_jshtml())

animate_forecasts(y_te, y_hat_lgb, y_hat_knn)

In [ ]:
df_red_diff = df_red.copy()
df_red_diff[[target]] = df_red_diff[[target]].diff().dropna()

# add target transform
y = add_lags(df_red_diff[[target]], [target], -np.arange(1, 25)).iloc[:, 1:]
X = add_lags(df_red_diff.drop(columns=[target]), selected_vars[1:], np.hstack([np.arange(1, 25), 24*2]))
df_postprocess = pd.concat({'X':X, 'y':y}, axis=1).dropna()

X_tr_diff, y_tr_diff = df_postprocess['X'].iloc[:n_tr], df_postprocess['y'].iloc[:n_tr]
X_te_diff, y_te_diff = df_postprocess['X'].iloc[n_tr:], df_postprocess['y'].iloc[n_tr:]


lgb_models = [LGBMRegressor(force_col_wise=True,verbose=0).fit(X_tr_diff, y_tr_diff[c].values) for c in y.columns]
y_hat_lgb_diff = pd.DataFrame({f'{target}_lag_{-l}': m.predict(X_te_diff) for l, m in enumerate(lgb_models, 1)}, index=y_te_diff.index)
y_hat_knn_diff = pd.DataFrame(KNeighborsRegressor().fit(X_tr_diff, y_tr_diff.values).predict(X_te_diff), index=y_te_diff.index)


In [ ]:
y_hat_lgb_rec = X_te[target].values.reshape(-1, 1) + y_hat_lgb_diff.cumsum(axis=1)
y_hat_knn_rec = X_te[target].values.reshape(-1, 1) + y_hat_knn_diff.cumsum(axis=1)
animate_forecasts(y_te, y_hat_lgb_rec, y_hat_knn_rec)

In [ ]:
plt.plot(np.mean((y_hat_knn-y_te.values)**2, axis=0).values**0.5, linestyle='--', label='knn')
plt.plot(np.mean((y_hat_lgb-y_te.values)**2, axis=0).values**0.5, linestyle='--', label='lgb')

plt.plot(np.mean((y_hat_knn_rec-y_te.values).values**2, axis=0)**0.5, label='knn diff')
plt.plot(np.mean((y_hat_lgb_rec-y_te.values).values**2, axis=0)**0.5, label='lgb diff')

plt.xlabel('forecast horizon')
plt.ylabel('rmse')
plt.legend()



In [ ]:
# plot ACF of residuals
fig, ax = plt.subplots(2, 1, figsize=(10, 4), layout='tight')
plot_acf((y_hat_lgb-y_te.values).values[:, 0], lags=24, ax=ax[0], linewidth=1);
plot_acf((y_hat_knn-y_te.values).values[:, 0], lags=24, ax=ax[1], linewidth=1);


plot_acf((y_hat_lgb_rec-y_te.values).values[:, 0], lags=24, ax=ax[0], linewidth=1);
plot_acf((y_hat_knn_rec-y_te.values).values[:, 0], lags=24, ax=ax[1], linewidth=1);


